# 4.1 The two-level phase

This notebook accompanies Section 4.1 of [*Community Detection with the Map Equation and Infomap: Theory and Applications*](https://doi.org/10.1145/3779648).

Infomap's two-level phase runs in two stages: the core algorithm builds a partition from scratch, then tuning refines it to escape local minima. Step through both stages on a small network and watch the partition change after each move.


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

import random

In [ ]:
import _support as util

In [ ]:
G, pos = util.get_map_equation_example_network()
util.plot_network(G)

## Stage 1: initialization

Infomap starts from the trivial partition that assigns each node to its own singleton module, which has a high codelength. Set up that starting point by hand and pass `no_infomap=True` to skip optimization.


In [ ]:
N = G.number_of_nodes()

partition = dict(zip(G.nodes, random.sample(range(N), N)))
util.partition(G, initial_partition=partition, no_infomap=True)
util.plot_network(G)

## Stage 1: core algorithm, first pass

The core algorithm visits the nodes in random order and moves each one to the neighboring module that reduces the codelength the most, or leaves it where it is. Run a single pass with `core_loop_limit=1`, hold off on tuning with `tune_iteration_limit=1`, and keep `core_level_limit` from re-running the core loop on the aggregated network.


In [ ]:
util.partition(
    G, two_level=True, core_loop_limit=1, core_level_limit=1, tune_iteration_limit=1, seed=123
)
util.plot_network(G)

## Stage 1: core algorithm, second pass

Moving a node marks it and its neighbors as candidates again, so a second pass over the candidates can still lower the codelength.


In [ ]:
util.partition(
    G, two_level=True, core_loop_limit=2, core_level_limit=1, tune_iteration_limit=1, seed=123
)
util.plot_network(G)

## Stage 1: core algorithm, third pass


In [ ]:
util.partition(
    G, two_level=True, core_loop_limit=3, core_level_limit=1, tune_iteration_limit=1, seed=123
)
util.plot_network(G)

## Stage 1: core algorithm, fourth pass


In [ ]:
util.partition(
    G, two_level=True, core_loop_limit=4, core_level_limit=1, tune_iteration_limit=1, seed=123
)
util.plot_network(G)

## Stage 1: moving modules as groups

The fourth pass found no improving move for any node. Infomap now switches to moving whole modules: it aggregates the network so each module becomes a node at level 2 and merges the links between them, then runs the core algorithm on that aggregated network. Nodes in the same module move together as a group, and the step repeats with larger and larger modules as long as the codelength keeps dropping.

A single core loop on the aggregated network already places the groups into their final modules.


In [ ]:
util.partition(
    G, two_level=True, core_loop_limit=1, core_level_limit=2, tune_iteration_limit=1, seed=123
)
util.plot_network(G)

## Stage 2: tuning

The core algorithm can settle in a local minimum. Tuning alternates fine-tuning, which moves individual nodes again, with coarse-tuning, which re-splits every module into sub-modules and moves those, until neither improves the codelength by the minimum amount. Run the full two-level algorithm and compare its partition with the core-only result above.


In [ ]:
util.partition(G, two_level=True, seed=123)
util.plot_network(G)